# 第10章 文件操作专题：读取、写入与目录管理

本章把第9章的路径函数应用到真实文件流程中，主线使用 `os`、`open`、`csv` 和 `json`。

## 本章学习目标

你将完成一个小型文件处理流程：创建目录、生成原始 CSV、逐行读取、校验记录、输出清洗文件、保存 JSON 摘要，并处理常见文件错误。

本章每个重要函数都使用独立 Cell，说明作用、参数、返回值、是否修改文件系统和边界情况。

## 1. `os.makedirs()`：准备输入和输出目录

In [ ]:
import os
import csv
import json

workspace = os.path.join(os.getcwd(), "chapter10_file_demo")
input_dir = os.path.join(workspace, "input")
output_dir = os.path.join(workspace, "output")
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
print("工作目录:", workspace)
print("输入目录:", input_dir)
print("输出目录:", output_dir)

## 2. `open(..., "w")` 与 `write()`：写入文本

- `open()` 返回文件对象；`w` 模式会覆盖旧内容。
- `write(text)` 返回写入字符数，不会自动换行。
- `encoding="utf-8"` 避免中文环境乱码。
- **副作用**：会创建或覆盖文件。
- **边界情况**：路径父目录不存在、没有权限或写入内容不是字符串。

In [ ]:
notes_path = os.path.join(input_dir, "notes.txt")
with open(notes_path, "w", encoding="utf-8") as file:
    count1 = file.write("第一行：输入文件\n")
    count2 = file.write("第二行：文件处理\n")
print("写入字符数:", count1 + count2)
print("文件存在:", os.path.isfile(notes_path))

## 3. `open(..., "r")`、`read()` 与 `readlines()`

- `read()` 返回剩余全部文本。
- `readlines()` 返回字符串列表，每个元素通常包含换行符。
- 文件对象的读取位置会向后移动；再次读取可能得到空字符串。
- **副作用**：不修改文件，但会消耗内存；大文件应逐行处理。

In [ ]:
with open(notes_path, "r", encoding="utf-8") as file:
    content = file.read()
print("完整内容:", repr(content))
with open(notes_path, "r", encoding="utf-8") as file:
    lines = file.readlines()
print("行数:", len(lines), "第一行:", repr(lines[0]))

## 4. `open(..., "a")`：追加写入

- **作用**：在文件末尾增加内容，不覆盖原内容。
- **参数**：路径、模式 `a`、编码。
- **返回值**：文件对象。
- **副作用**：修改文件。
- **边界情况**：追加内容仍需自行添加换行；重复运行会重复追加。

In [ ]:
with open(notes_path, "a", encoding="utf-8") as file:
    file.write("第三行：追加内容\n")
print(open(notes_path, encoding="utf-8").read())

## 5. `csv.writer()` 与 `writerow()`：写出表格文件

- `csv.writer(file)` 创建 CSV 写入器。
- `writerow(row)` 写入一行，返回写入字段数。
- `writerows(rows)` 一次写入多行。
- **副作用**：修改文件。
- **边界情况**：Windows 环境建议 `newline=""`；字段中包含逗号时交给 CSV 模块处理。

In [ ]:
records = [["name", "score"], ["Alice", 91], ["Bob", 84], ["Cathy", 96]]
csv_path = os.path.join(input_dir, "scores.csv")
with open(csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    print("表头字段数:", writer.writerow(records[0]))
    writer.writerows(records[1:])
print("CSV 文件:", csv_path, "大小:", os.path.getsize(csv_path))

## 6. `csv.DictReader()`：按字段名读取 CSV

- **作用**：把每一行转换成字典。
- **参数**：文件对象；默认使用首行作为字段名。
- **返回值**：可迭代对象，每行是字典。
- **是否修改文件**：不修改。
- **边界情况**：表头缺失、字段数量不一致和空文件都需要单独处理。

In [ ]:
with open(csv_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    loaded = list(reader)
print("读取记录:", loaded)
print("Alice 分数:", loaded[0]["score"])

## 7. `json.dump()` 与 `json.load()`：保存结构化结果

- `json.dump(data, file)` 把 Python 对象写入文件。
- `json.load(file)` 从文件读取 JSON 对象。
- 常用参数 `ensure_ascii=False` 保留中文，`indent=2` 便于阅读。
- **副作用**：`dump()` 会修改文件，`load()` 不修改。
- **边界情况**：对象不可序列化、文件为空或 JSON 格式错误。

In [ ]:
summary = {"count": len(loaded), "mean": sum(int(row["score"]) for row in loaded) / len(loaded)}
json_path = os.path.join(output_dir, "summary.json")
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)
with open(json_path, "r", encoding="utf-8") as file:
    loaded_summary = json.load(file)
print("JSON 摘要:", loaded_summary)

## 8. 自定义函数：读取、校验和生成报告

下面把前面学过的函数组合成业务流程。每个函数仍然只负责一个任务。

### `read_score_rows(path)`

- **作用**：读取 CSV 并返回字典列表。
- **参数**：CSV 文件路径。
- **返回值**：列表。
- **副作用**：不修改文件。
- **边界情况**：文件不存在、空文件、缺少 `name` 或 `score` 字段。

In [ ]:
def read_score_rows(path):
    with open(path, "r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))

rows = read_score_rows(csv_path)
print("函数读取行数:", len(rows))

### `validate_score_rows(rows)`

- **作用**：检查字段是否存在、分数能否转换并返回合格行和问题行。
- **参数**：字典列表。
- **返回值**：`(valid_rows, invalid_rows)` 二元组。
- **副作用**：不修改原列表。
- **边界情况**：空列表、缺字段、空字符串和分数超出范围。

In [ ]:
def validate_score_rows(rows):
    valid, invalid = [], []
    for number, row in enumerate(rows, start=2):
        try:
            score = float(row["score"])
            if not 0 <= score <= 100: raise ValueError("分数超出范围")
            valid.append({"name": row["name"], "score": score})
        except (KeyError, TypeError, ValueError) as error:
            invalid.append({"line": number, "row": row, "reason": str(error)})
    return valid, invalid

valid_rows, invalid_rows = validate_score_rows(rows)
print("合格:", valid_rows, "问题:", invalid_rows)

### `write_json_report(path, valid_rows, invalid_rows)`

- **作用**：生成质量和统计摘要 JSON。
- **参数**：输出路径、合格行列表、问题行列表。
- **返回值**：返回写入的摘要字典。
- **副作用**：创建或覆盖 JSON 文件。
- **边界情况**：输出目录不存在、没有合格记录、路径无写权限。

In [ ]:
def write_json_report(path, valid_rows, invalid_rows):
    scores = [row["score"] for row in valid_rows]
    report = {"valid_count": len(valid_rows), "invalid_count": len(invalid_rows), "mean_score": round(sum(scores)/len(scores), 2) if scores else None}
    with open(path, "w", encoding="utf-8") as file:
        json.dump(report, file, ensure_ascii=False, indent=2)
    return report

report = write_json_report(os.path.join(output_dir, "score_report.json"), valid_rows, invalid_rows)
print(report)

## 9. `os.walk()`：递归遍历目录

- **作用**：递归生成目录、子目录和文件名。
- **参数**：起始目录。
- **返回值**：可迭代对象，每次返回 `(root, dirs, files)`。
- **是否修改文件**：不修改。
- **边界情况**：目录不存在、权限不足和符号链接循环。

In [ ]:
for current_root, dirs, files in os.walk(workspace):
    print("目录:", current_root)
    print("文件:", files)

## 10. 文件错误处理

文件操作失败是可预期情况。用具体异常和明确提示代替静默失败。

In [ ]:
def safe_read(path):
    try:
        with open(path, "r", encoding="utf-8") as file:
            return {"ok": True, "content": file.read(), "message": "读取成功"}
    except FileNotFoundError:
        return {"ok": False, "content": "", "message": "文件不存在"}
    except UnicodeError:
        return {"ok": False, "content": "", "message": "编码无法识别"}

print(safe_read(notes_path)["message"])
print(safe_read(os.path.join(input_dir, "missing.csv"))["message"])

## 11. 本章提交练习

1. 将成绩 CSV 清洗后写入 `clean_scores.csv`。
2. 将问题行写入 `invalid_scores.csv`。
3. 生成包含记录数、均值、最大值、最小值的 JSON。
4. 用 `os.walk()` 统计目录中每种扩展名的文件数量。
5. 写出覆盖写入、追加写入、读取和目录遍历各自的适用场景。

## 12. 本章完成检查

- [ ] 每个重要函数都有独立说明和独立 Cell。
- [ ] 能说明参数、返回值、文件系统副作用和边界情况。
- [ ] 能使用 os、open、csv 和 json 完成小型文件流程。
- [ ] 文件错误有明确提示。
- [ ] 没有使用 断言语句 做教学自检。